In [1]:
import os
import re
import time
import json
import glob
import requests
import pandas as pd
from difflib import SequenceMatcher

BASE = "/Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Processed Data"
COMBINED_PAST = os.path.join(BASE, "Combined Past Holdings")

PAST_FILES = [os.path.join(COMBINED_PAST, f"soi_{y}.csv") for y in range(2017, 2025)]
HOLDINGS_2025 = os.path.join(BASE, "holdings_2025.csv")
CC_GLOB = os.path.join(BASE, "Controversial_and_Clean_Holdings*.xlsx")

UNMATCHED_LOG = os.path.join(BASE, "unmatched_names_all_files_yahoo_only.csv")
CACHE_PATH = os.path.join(BASE, "name_to_ticker_cache_yahoo_only.json")

YAHOO_SEARCH_URL = "https://query2.finance.yahoo.com/v1/finance/search"
YAHOO_SLEEP_SECONDS = 0.4    
YAHOO_TIMEOUT = 8.0

PREFERRED_EXCHANGES = {"NASDAQ", "NYSE", "NYSE ARCA", "NYSE MKT", "NASDAQGS", "NASDAQCM", "NASDAQGM"}

MIN_SIMILARITY = 0.55  

NAME_CANDIDATES = [
    "security", "holding", "holding_name", "name", "company", "company_name", "holding name"
]

def normalize_company_name(s: str) -> str:
    """Uppercase, strip punctuation and common suffixes (INC, LTD, PLC, CORP, GROUP, ADR/ADS, CLASS A/B/C)."""
    if pd.isna(s):
        return ""
    s = str(s).upper()
    s = re.sub(r"[^\w\s]", " ", s)
    suffixes = [
        r"\bINC\b", r"\bINCORPORATED\b", r"\bLTD\b", r"\bLIMITED\b",
        r"\bPLC\b", r"\bCORP\b", r"\bCORPORATION\b", r"\bNV\b", r"\bAG\b",
        r"\bSA\b", r"\bS A\b", r"\bCO\b", r"\bCOMPANY\b", r"\bHLDGS\b",
        r"\bHOLDINGS\b", r"\bHLDG\b", r"\bGROUP\b",
        r"\bCLASS\s*[A-Z]\b", r"\bCL\s*[A-Z]\b",
        r"\bSPONSORED\b", r"\bADR\b", r"\bADS\b"
    ]
    s = re.sub("|".join(suffixes), " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def find_name_column(df: pd.DataFrame) -> str:
    """Pick the most likely company-name column."""
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in NAME_CANDIDATES:
        if cand in cols_lower:
            return cols_lower[cand]
    for c in df.columns:  # last resort
        if "name" in c.lower():
            return c
    raise ValueError(f"Could not find a company-name column. Tried: {', '.join(NAME_CANDIDATES)}")


def similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, a, b).ratio()


def choose_best_quote(query_original: str, quotes: list) -> str:
    """
    Score Yahoo quotes and return the best symbol, or '' if none meet MIN_SIMILARITY.
    Scoring heuristics:
      - Similarity between normalized names
      - Prefer EQUITY quoteType
      - Prefer major US exchanges
    """
    q_norm = normalize_company_name(query_original)
    best = None
    best_score = -1.0

    for q in quotes:
        symbol = (q.get("symbol") or "").strip()
        if not symbol or len(symbol) > 12:
            continue

        qt = (q.get("quoteType") or "").upper()
        exch = (q.get("exchDisp") or q.get("exchange") or "").upper()
        yname = (q.get("shortname") or q.get("longname") or "").strip()
        y_norm = normalize_company_name(yname)

        sim = similarity(q_norm, y_norm)

        # scoring
        score = sim * 10.0 
        if qt == "EQUITY":
            score += 2.5
        if exch in PREFERRED_EXCHANGES:
            score += 1.5
        if q_norm and y_norm and (q_norm in y_norm or y_norm in q_norm):
            score += 1.0

        if score > best_score:
            best_score = score
            best = (symbol, sim)

    if best is None:
        return ""
    symbol, sim = best
    return symbol if sim >= MIN_SIMILARITY else ""


def yahoo_lookup(session: requests.Session, company_name: str) -> str:
    """Query Yahoo search for a likely ticker; return '' if not confident/available."""
    try:
        params = {"q": company_name, "quotesCount": 10, "newsCount": 0, "lang": "en-US", "region": "US"}
        r = session.get(YAHOO_SEARCH_URL, params=params, timeout=YAHOO_TIMEOUT)
        if r.status_code != 200:
            return ""
        data = r.json()
        quotes = data.get("quotes", []) or []
        if not quotes:
            return ""
        return choose_best_quote(company_name, quotes)
    except Exception:
        return ""


def load_cache(path: str) -> dict:
    if os.path.exists(path):
        try:
            with open(path, "r") as f:
                return json.load(f)
        except Exception:
            return {}
    return {}


def save_cache(path: str, cache: dict):
    try:
        with open(path, "w") as f:
            json.dump(cache, f, indent=2, sort_keys=True)
    except Exception:
        pass


def enrich_csv_with_yahoo(file_path: str, cache: dict, session: requests.Session) -> tuple[str, pd.DataFrame]:
    """Add company_ticker to a CSV via Yahoo lookup. Returns (output_path, df)."""
    df = pd.read_csv(file_path)
    name_col = find_name_column(df)
    df["name_normalized"] = df[name_col].apply(normalize_company_name)

    representatives = df.groupby("name_normalized")[name_col].apply(lambda x: str(x.dropna().iloc[0]) if not x.dropna().empty else "").to_dict()

    results = {}
    for nm, original in representatives.items():
        if not nm:
            results[nm] = ""
            continue
        key = f"{nm}||{original.strip()}"
        if key in cache:
            results[nm] = cache[key]
            continue

        sym = yahoo_lookup(session, original if original else nm)
        results[nm] = sym
        cache[key] = sym
        time.sleep(YAHOO_SLEEP_SECONDS)

    df["company_ticker"] = df["name_normalized"].map(results).fillna("")
    out_path = os.path.splitext(file_path)[0] + "_with_tickers.csv"
    df.to_csv(out_path, index=False)
    return out_path, df


def enrich_excel_with_yahoo(file_path: str, cache: dict, session: requests.Session) -> tuple[str, pd.DataFrame]:
    """Add company_ticker to FIRST sheet of Excel via Yahoo lookup. Returns (output_path, df)."""
    xls = pd.ExcelFile(file_path)
    sheet = xls.sheet_names[0]
    df = pd.read_excel(file_path, sheet_name=sheet)

    name_col = find_name_column(df)
    df["name_normalized"] = df[name_col].apply(normalize_company_name)

    representatives = df.groupby("name_normalized")[name_col].apply(lambda x: str(x.dropna().iloc[0]) if not x.dropna().empty else "").to_dict()

    results = {}
    for nm, original in representatives.items():
        if not nm:
            results[nm] = ""
            continue
        key = f"{nm}||{original.strip()}"
        if key in cache:
            results[nm] = cache[key]
            continue

        sym = yahoo_lookup(session, original if original else nm)
        results[nm] = sym
        cache[key] = sym
        time.sleep(YAHOO_SLEEP_SECONDS)

    df["company_ticker"] = df["name_normalized"].map(results).fillna("")
    out_path = os.path.splitext(file_path)[0] + "_with_tickers.xlsx"
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name=sheet)
    return out_path, df

def main():
    cache = load_cache(CACHE_PATH)
    unmatched_frames = []

    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (compatible; ESGTickerBot/1.0; +https://example.org)",
        "Accept": "application/json,text/javascript,*/*;q=0.1",
        "Connection": "keep-alive",
    })

    for f in PAST_FILES:
        if not os.path.exists(f):
            print(f"[WARN] Missing: {f}")
            continue
        try:
            out, df = enrich_csv_with_yahoo(f, cache, session)
            print(f"[OK] Wrote: {out}")
            um = df.loc[df["company_ticker"].eq(""), ["name_normalized"]].drop_duplicates()
            um["source_file"] = os.path.basename(f)
            unmatched_frames.append(um)
        except Exception as e:
            print(f"[ERROR] {f}: {e}")

    if os.path.exists(HOLDINGS_2025):
        try:
            out, df = enrich_csv_with_yahoo(HOLDINGS_2025, cache, session)
            print(f"[OK] Wrote: {out}")
            um = df.loc[df["company_ticker"].eq(""), ["name_normalized"]].drop_duplicates()
            um["source_file"] = os.path.basename(HOLDINGS_2025)
            unmatched_frames.append(um)
        except Exception as e:
            print(f"[ERROR] {HOLDINGS_2025}: {e}")
    else:
        print(f"[WARN] Missing: {HOLDINGS_2025}")

    cc_paths = sorted(glob.glob(CC_GLOB))
    if not cc_paths:
        print(f"[WARN] Could not find Excel by pattern: {CC_GLOB}")
    else:
        cc_file = cc_paths[0]
        try:
            out, df = enrich_excel_with_yahoo(cc_file, cache, session)
            print(f"[OK] Wrote: {out}")
            um = df.loc[df["company_ticker"].eq(""), ["name_normalized"]].drop_duplicates()
            um["source_file"] = os.path.basename(cc_file)
            unmatched_frames.append(um)
        except Exception as e:
            print(f"[ERROR] {cc_file}: {e}")

    if unmatched_frames:
        all_um = pd.concat(unmatched_frames, ignore_index=True).drop_duplicates().sort_values(["source_file", "name_normalized"])
        all_um.to_csv(UNMATCHED_LOG, index=False)
        print(f"[INFO] Unmatched log: {UNMATCHED_LOG}")
    else:
        print("[INFO] No unmatched names (great).")

    save_cache(CACHE_PATH, cache)
    print(f"[INFO] Cache saved: {CACHE_PATH}")


if __name__ == "__main__":
    main()


[OK] Wrote: /Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings/soi_2017_with_tickers.csv
[OK] Wrote: /Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings/soi_2018_with_tickers.csv
[OK] Wrote: /Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings/soi_2019_with_tickers.csv
[OK] Wrote: /Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings/soi_2020_with_tickers.csv
[OK] Wrote: /Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings/soi_2021_with_tickers.csv
[OK] Wrote: /Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings/soi_2022_with_tickers.csv
[OK] Wrote: /Users/nityaarya/Downloads/blackrock-esg-etf-study/b

In [3]:
import os
import re
import glob
import json
import time
import hashlib
import requests
import pandas as pd
from difflib import SequenceMatcher

BASE = "/Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Processed Data"
COMBINED_PAST = os.path.join(BASE, "Combined Past Holdings")

PAST_FILES = [os.path.join(COMBINED_PAST, f"soi_{y}.csv") for y in range(2017, 2025)]
HOLDINGS_2025 = os.path.join(BASE, "holdings_2025.csv")
CC_GLOB = os.path.join(BASE, "Controversial_and_Clean_Holdings*.xlsx") 

FINAL_MAP_JSON = os.path.join(BASE, "final_name_to_ticker_map.json") 
RUN_REPORT_CSV = os.path.join(BASE, "zero_gap_resolution_report.csv")

YAHOO_SEARCH_URL = "https://query2.finance.yahoo.com/v1/finance/search"
YAHOO_SLEEP = 0.35
YAHOO_TIMEOUT = 8.0
MIN_SIMILARITY = 0.58
PREFERRED_EXCHANGES = {"NASDAQ","NASDAQGS","NASDAQGM","NASDAQCM","NYSE","NYSE ARCA","NYSE MKT"}

NAME_CANDIDATES = ["security","holding","holding_name","name","company","company_name","holding name"]

def normalize_company_name(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).upper()
    s = re.sub(r"[^\w\s]", " ", s)
    suffixes = [
        r"\bINC\b", r"\bINCORPORATED\b", r"\bLTD\b", r"\bLIMITED\b",
        r"\bPLC\b", r"\bCORP\b", r"\bCORPORATION\b", r"\bNV\b", r"\bAG\b",
        r"\bSA\b", r"\bS A\b", r"\bCO\b", r"\bCOMPANY\b", r"\bHLDGS\b",
        r"\bHOLDINGS\b", r"\bHLDG\b", r"\bGROUP\b",
        r"\bCLASS\s*[A-Z]\b", r"\bCL\s*[A-Z]\b",
        r"\bSPONSORED\b", r"\bADR\b", r"\bADS\b"
    ]
    s = re.sub("|".join(suffixes), " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def find_name_col(df: pd.DataFrame) -> str:
    lower = {c.lower(): c for c in df.columns}
    for cand in NAME_CANDIDATES:
        if cand in lower:
            return lower[cand]
    for c in df.columns:
        if "name" in c.lower():
            return c
    raise ValueError(f"Could not find a company-name column in: {list(df.columns)}")

def sim(a: str, b: str) -> float:
    return SequenceMatcher(None, a, b).ratio()

def choose_best_quote(query_original: str, quotes: list) -> str:
    q_norm = normalize_company_name(query_original)
    best, best_score = None, -1.0
    for q in quotes:
        symbol = (q.get("symbol") or "").strip()
        if not symbol or len(symbol) > 12:
            continue
        qt = (q.get("quoteType") or "").upper()
        exch = (q.get("exchDisp") or q.get("exchange") or "").upper()
        yname = (q.get("shortname") or q.get("longname") or "").strip()
        y_norm = normalize_company_name(yname)
        s = sim(q_norm, y_norm)
        score = s * 10.0
        if qt == "EQUITY":
            score += 2.5
        if exch in PREFERRED_EXCHANGES:
            score += 1.5
        if q_norm and y_norm and (q_norm in y_norm or y_norm in q_norm):
            score += 1.0
        if score > best_score:
            best_score, best = score, (symbol, s)
    if best is None:
        return ""
    sym, s = best
    return sym if s >= MIN_SIMILARITY else ""

def yahoo_lookup(session: requests.Session, name: str) -> str:
    try:
        params = {"q": name, "quotesCount": 10, "newsCount": 0, "lang": "en-US", "region": "US"}
        r = session.get(YAHOO_SEARCH_URL, params=params, timeout=YAHOO_TIMEOUT)
        if r.status_code != 200:
            return ""
        quotes = (r.json() or {}).get("quotes", []) or []
        if not quotes:
            return ""
        return choose_best_quote(name, quotes)
    except Exception:
        return ""

def mint_synthetic_ticker(name_norm: str, used: set) -> str:
    """
    Create a deterministic placeholder ticker for a normalized name.
    - Prefix 'ZZ_' to signal synthetic
    - Base: letters from name_norm, take consonant-ish acronym; fallback to hash.
    - Ensure uniqueness by appending digits if needed.
    """
    base = re.sub(r"[^A-Z0-9 ]", "", name_norm)
    words = [w for w in base.split() if w]
    
    acronym = "".join(w[0] for w in words)[:5]
    if len(acronym) < 3:
        # fall back to 4 chars from sha1
        h = hashlib.sha1(name_norm.encode("utf-8")).hexdigest().upper()
        acronym = (name_norm[:2] + h[:3]).upper()
        acronym = re.sub(r"[^A-Z0-9]", "", acronym)
    ticker = f"ZZ_{acronym}"
    
    if ticker not in used:
        return ticker
    
    h2 = hashlib.md5(name_norm.encode("utf-8")).hexdigest().upper()
    for k in range(1, 10):
        cand = f"{ticker}{k}"
        if cand not in used:
            return cand
    
    for sl in range(2, 8):
        cand = f"{ticker}_{h2[:sl]}"
        if cand not in used:
            return cand
    # if somehow still not unique:
    i = 1
    while True:
        cand = f"{ticker}_{i}"
        if cand not in used:
            return cand
        i += 1

def main():
    
    csv_targets = [f for f in PAST_FILES if os.path.exists(f)]
    if os.path.exists(HOLDINGS_2025):
        csv_targets.append(HOLDINGS_2025)
    xlsx_targets = sorted(glob.glob(CC_GLOB))

    if os.path.exists(FINAL_MAP_JSON):
        with open(FINAL_MAP_JSON, "r") as f:
            final_map = json.load(f)  # dict[name_normalized] = ticker
    else:
        final_map = {}

    name_info = {}  
    def ingest_df(df, name_col, ticker_col):
        for _, row in df.iterrows():
            name_raw = str(row.get(name_col, "")).strip()
            nm = normalize_company_name(name_raw)
            if not nm:
                continue
            tkr = str(row.get(ticker_col, "")).strip() if ticker_col in df.columns else ""
            if nm not in name_info:
                name_info[nm] = {"examples": set(), "tickers": set()}
            if name_raw:
                name_info[nm]["examples"].add(name_raw)
            if tkr:
                name_info[nm]["tickers"].add(tkr)

    for f in csv_targets:
        df = pd.read_csv(f)
        name_col = find_name_col(df)
        ticker_col = None
        for c in df.columns:
            if c.lower() in ["ticker","company_ticker","symbol"]:
                ticker_col = c
                break
        if ticker_col is None:
            df["company_ticker"] = ""
            ticker_col = "company_ticker"
        ingest_df(df, name_col, ticker_col)

    if xlsx_targets:
        cc_file = xlsx_targets[0]
        xls = pd.ExcelFile(cc_file)
        sheet = xls.sheet_names[0]
        cc = pd.read_excel(cc_file, sheet_name=sheet)
        name_col = find_name_col(cc)
        ticker_col = None
        for c in cc.columns:
            if c.lower() in ["ticker","company_ticker","symbol"]:
                ticker_col = c
                break
        if ticker_col is None:
            cc["company_ticker"] = ""
            ticker_col = "company_ticker"
        ingest_df(cc, name_col, ticker_col)

    for nm, info in name_info.items():
        tkrs = [t for t in info["tickers"] if t]  # non-empty
        if len(set(tkrs)) == 1:
            final_map.setdefault(nm, tkrs[0])

    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0 (compatible; ZeroGapBot/1.0)"})

    used_tickers = set(t for v in name_info.values() for t in v["tickers"] if t)
    used_tickers.update(final_map.values())

    operations = [] 

    for nm, info in sorted(name_info.items()):
        if final_map.get(nm): 
            operations.append((nm, final_map[nm], "existing"))
            continue

        original = next(iter(info["examples"])) if info["examples"] else nm
        sym = yahoo_lookup(session, original if original else nm)
        if sym:
            final_map[nm] = sym
            used_tickers.add(sym)
            operations.append((nm, sym, "yahoo"))
        else:
            syn = mint_synthetic_ticker(nm, used_tickers)
            final_map[nm] = syn
            used_tickers.add(syn)
            operations.append((nm, syn, "synthetic"))
        time.sleep(YAHOO_SLEEP)

    with open(FINAL_MAP_JSON, "w") as f:
        json.dump(final_map, f, indent=2, sort_keys=True)
    print(f"[INFO] Saved final name->ticker map: {FINAL_MAP_JSON}")

    def apply_and_write_csv(path):
        df = pd.read_csv(path)
        name_col = find_name_col(df)
        if "company_ticker" not in df.columns and "ticker" not in df.columns and "symbol" not in df.columns:
            df["company_ticker"] = ""
            ticker_col = "company_ticker"
        else:
            ticker_col = "company_ticker" if "company_ticker" in df.columns else \
                         ("ticker" if "ticker" in df.columns else "symbol")
        df["name_normalized"] = df[name_col].apply(normalize_company_name)
        df[ticker_col] = df["name_normalized"].map(final_map).fillna(df.get(ticker_col, ""))  # no gaps
        out = os.path.splitext(path)[0] + "_final.csv"
        df.to_csv(out, index=False)
        print(f"[OK] wrote {out}")

    for f in csv_targets:
        apply_and_write_csv(f)

    def apply_and_write_excel(path):
        xls = pd.ExcelFile(path)
        sheet = xls.sheet_names[0]
        df = pd.read_excel(path, sheet_name=sheet)
        name_col = find_name_col(df)
        if "company_ticker" not in df.columns and "ticker" not in df.columns and "symbol" not in df.columns:
            df["company_ticker"] = ""
            ticker_col = "company_ticker"
        else:
            ticker_col = "company_ticker" if "company_ticker" in df.columns else \
                         ("ticker" if "ticker" in df.columns else "symbol")
        df["name_normalized"] = df[name_col].apply(normalize_company_name)
        df[ticker_col] = df["name_normalized"].map(final_map).fillna(df.get(ticker_col, ""))  # no gaps
        out = os.path.splitext(path)[0] + "_final.xlsx"
        with pd.ExcelWriter(out, engine="openpyxl") as writer:
            df.to_excel(writer, index=False, sheet_name=sheet)
        print(f"[OK] wrote {out}")

    if xlsx_targets:
        apply_and_write_excel(xlsx_targets[0])

    rep = pd.DataFrame(operations, columns=["name_normalized","resolved_ticker","source"])
    rep.to_csv(RUN_REPORT_CSV, index=False)
    print(f"[INFO] Resolution report: {RUN_REPORT_CSV}")
    print("[DONE] All gaps filled. Every name now has a ticker (real or synthetic).")


if __name__ == "__main__":
    main()


[INFO] Saved final name->ticker map: /Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Processed Data/final_name_to_ticker_map.json
[OK] wrote /Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings/soi_2017_final.csv
[OK] wrote /Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings/soi_2018_final.csv
[OK] wrote /Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings/soi_2019_final.csv
[OK] wrote /Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings/soi_2020_final.csv
[OK] wrote /Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings/soi_2021_final.csv
[OK] wrote /Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Process